In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow_hub as hub
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Concatenate, Input, MaxPooling2D, Flatten, Conv2D, Dense, BatchNormalization, GlobalAveragePooling2D, Dropout, AveragePooling2D, Resizing, Activation, Reshape, Multiply
from tensorflow.keras.applications.densenet import DenseNet169
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from sklearn.metrics import accuracy_score
from tensorflow.keras.applications.regnet import RegNetY320
import pandas as pd
import os
from tensorflow.keras.preprocessing.image import img_to_array, load_img, array_to_img
import skimage
from skimage import io
import cv2
import warnings
warnings.simplefilter(action = 'ignore', category = FutureWarning)

## Loading the data

In [2]:
def load_data(csv_dir, mode, first = 'Yes'):
    angles = [30, 60, 90, 120, 150, 180, 210, 240, 270, 300]
    df = pd.read_csv(csv_dir, header = None)
    df[1] = (df[1] - 1).astype(str)
    df[0] = df[0].astype(str)
    df[0] = 'Dataset/' + mode + '/' + df[0] + '.png'
    df.columns = ['filename', 'class']
    if mode == 'Train' and first == 'Yes':
        for i in tqdm(range(len(df['filename']))):
            image = df['filename'][i]
            label = df['class'][i]
            img = img_to_array(load_img(image, target_size = (224, 224)))
            img = img / 255.0
            for angle in angles:
                gimg = skimage.transform.rotate(img, angle)
                gimg = array_to_img(gimg)
                new_path = image[0 : 14] + str(angle) + '_' + image[14 : ]
                gimg.save(new_path)
                df = df.append({'filename': new_path, 'class': label}, ignore_index = True)
        df.to_csv('Dataset/Updated_Train.csv', index = False)
        
    return df

In [3]:
# df_train = load_data('Dataset/Train.csv', 'Train', first = 'No')
df_train = pd.read_csv('Dataset/Updated_Train.csv')
df_train['class'] = df_train['class'].astype(str)
df_valid = load_data('Dataset/Validation.csv', 'Validation', first = 'No')

In [4]:
grouped_train = df_train.groupby('class')
shifted_examples = pd.DataFrame(columns = df_train.columns)
for group_name, group_df in grouped_train:
    shifted_examples = pd.concat([shifted_examples, group_df.tail(50)])
df_train = df_train.drop(shifted_examples.index)
df_valid = pd.concat([df_valid, shifted_examples])

## Applying preprocessing and creating the Data Generators

In [5]:
image_gen = ImageDataGenerator(
    rescale = 1. / 255,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    zoom_range = 0.2,
    shear_range = 0.2,
    fill_mode = 'nearest'
)
train_set = image_gen.flow_from_dataframe(
    df_train,
    target_size = (224, 224),
    batch_size = 16,
    class_mode = 'categorical',
)
valid_set = image_gen.flow_from_dataframe(
    df_valid,
    target_size = (224, 224),
    batch_size = 16,
    class_mode = 'categorical'
)

Found 97500 validated image filenames belonging to 250 classes.
Found 17500 validated image filenames belonging to 250 classes.


## Model Creation and Training

In [6]:
def attention_block(input_feature, ratio = 8):
    channel = input_feature.shape[-1]
    squeeze = GlobalAveragePooling2D()(input_feature)
    excitation = Dense(channel // ratio, activation = 'relu')(squeeze)
    excitation = Dense(channel, activation = 'sigmoid')(excitation)
    excitation = Reshape((1, 1, channel))(excitation)
    scale = Multiply()([input_feature, excitation])
    return scale

## First Model

In [7]:
# in_shape = (224, 224, 3)
# in_shape_1 = (224, 224, 3)
# in_shape_2 = (224, 224, 3)

# in_layer = Input(shape = in_shape)
# in_layer_1 = Resizing(224, 224)(in_layer)
# in_layer_2 = Resizing(224, 224)(in_layer)

# densenet = DenseNet169(weights = 'imagenet', input_shape = in_shape, include_top = False)
# mobilenet = MobileNetV2(weights = 'imagenet', input_shape = in_shape_1, include_top = False)
# efficientnet = EfficientNetB7(weights = 'imagenet', input_shape = in_shape_2, include_top = False)

# for layer in mobilenet.layers[0 : -(len(mobilenet.layers) // 4)]:
#     layer.trainable = False
# for layer in densenet.layers[0 : -(len(densenet.layers) // 4)]:
#     layer.trainable = False
# for layer in efficientnet.layers[0 : -(len(efficientnet.layers) // 4)]:
#     layer.trainable = False
    
# model_densenet = densenet(in_layer)
# output_densenet_attention = attention_block(model_densenet)
# output_densenet = GlobalAveragePooling2D()(output_densenet_attention)
# output_densenet = Flatten()(output_densenet)

# model_mobilenet = mobilenet(in_layer_1)
# output_mobilenet_attention = attention_block(model_mobilenet)
# output_mobilenet = GlobalAveragePooling2D()(output_mobilenet_attention)
# output_mobilenet = Flatten()(output_mobilenet)

# model_efficientnet = efficientnet(in_layer_2)
# output_efficientnet_attention = attention_block(model_efficientnet)
# output_efficientnet = GlobalAveragePooling2D()(output_efficientnet_attention)
# output_efficientnet = Flatten()(output_efficientnet)

# concatenated = Concatenate()([output_densenet, output_mobilenet, output_efficientnet])
# X = BatchNormalization()(concatenated)
# X = Dense(4096, activation = 'relu')(X)
# X = Dropout(0.5)(X)
# X = Dense(4096, activation = 'relu')(X)
# X = Dropout(0.5)(X)
# X = Dense(250, activation = 'linear')(X)

# siamese_model = tf.keras.models.Model(inputs = in_layer, outputs = X)

## Second Model

## Run from here

In [8]:
in_shape = (224, 224, 3)
in_shape_1 = (224, 224, 3)
in_shape_2 = (224, 224, 3)

in_layer = Input(shape = in_shape)
in_layer_1 = Resizing(224, 224)(in_layer)
in_layer_2 = Resizing(224, 224)(in_layer)

densenet = DenseNet169(weights = 'imagenet', input_shape = in_shape, include_top = False)
mobilenet = MobileNetV2(weights = 'imagenet', input_shape = in_shape_1, include_top = False)
regnet = RegNetY320(weights = 'imagenet', input_shape = in_shape_2, include_top = False)

for layer in regnet.layers[0 : -(len(regnet.layers) // 16)]:
    layer.trainable = False
for layer in densenet.layers[0 : -(len(densenet.layers) // 3)]:
    layer.trainable = False
for layer in mobilenet.layers[0 : -(len(mobilenet.layers) // 3)]:
    layer.trainable = False
    
model_densenet = densenet(in_layer)
output_densenet_attention = attention_block(model_densenet)
output_densenet = GlobalAveragePooling2D()(output_densenet_attention)
output_densenet = Flatten()(output_densenet)

model_regnet = regnet(in_layer_1)
output_regnet_attention = attention_block(model_regnet)
output_regnet = GlobalAveragePooling2D()(output_regnet_attention)
output_regnet = Flatten()(output_regnet)

model_mobilenet = mobilenet(in_layer_2)
output_mobilenet_attention = attention_block(model_mobilenet)
output_mobilenet = GlobalAveragePooling2D()(output_mobilenet_attention)
output_mobilenet = Flatten()(output_mobilenet)

concatenated = Concatenate()([output_densenet, output_regnet, output_mobilenet])
X = BatchNormalization()(concatenated)
X = Dense(8192, activation = 'relu')(X)
X = Dropout(0.5)(X)
X = Dense(8196, activation = 'relu')(X)
X = Dropout(0.5)(X)
X = Dense(250, activation = 'linear')(X)

siamese_model = tf.keras.models.Model(inputs = in_layer, outputs = X)

In [9]:
siamese_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 resizing (Resizing)            (None, 224, 224, 3)  0           ['input_1[0][0]']                
                                                                                                  
 resizing_1 (Resizing)          (None, 224, 224, 3)  0           ['input_1[0][0]']                
                                                                                                  
 densenet169 (Functional)       (None, 7, 7, 1664)   12642880    ['input_1[0][0]']            

 dropout (Dropout)              (None, 8192)         0           ['dense_6[0][0]']                
                                                                                                  
 dense_7 (Dense)                (None, 8196)         67149828    ['dropout[0][0]']                
                                                                                                  
 dropout_1 (Dropout)            (None, 8196)         0           ['dense_7[0][0]']                
                                                                                                  
 dense_8 (Dense)                (None, 250)          2049250     ['dropout_1[0][0]']              
                                                                                                  
Total params: 284,706,816
Trainable params: 170,047,490
Non-trainable params: 114,659,326
__________________________________________________________________________________________________


In [10]:
rlr = ReduceLROnPlateau(monitor = "val_accuracy", factor = 0.01, patience = 3, verbose = 0, mode = "max", min_delta = 0.01)

## Till the next one to define the model first

In [11]:
optim = Adam(learning_rate = 0.0001)
siamese_model.compile(loss = tf.keras.losses.CategoricalCrossentropy(from_logits = True), optimizer = optim, metrics = ['accuracy'])

In [18]:
history = siamese_model.fit(train_set, steps_per_epoch = 97500 // 16, epochs = 100, batch_size = 16, validation_data = valid_set, callbacks = [rlr])

Epoch 1/100
6093/6093 [==============================] - 4945s 812ms/step - loss: 0.0274 - accuracy: 0.9919 - val_loss: 1.1095 - val_accuracy: 0.8101 - lr: 5.0000e-07
Epoch 2/100
6093/6093 [==============================] - 4933s 810ms/step - loss: 0.0282 - accuracy: 0.9914 - val_loss: 1.0969 - val_accuracy: 0.8107 - lr: 5.0000e-07
Epoch 3/100
6093/6093 [==============================] - 4876s 800ms/step - loss: 0.0259 - accuracy: 0.9922 - val_loss: 1.1057 - val_accuracy: 0.8093 - lr: 5.0000e-07
Epoch 4/100
6093/6093 [==============================] - 4817s 790ms/step - loss: 0.0232 - accuracy: 0.9927 - val_loss: 1.0878 - val_accuracy: 0.8141 - lr: 5.0000e-07
Epoch 5/100
6093/6093 [==============================] - 4929s 809ms/step - loss: 0.0253 - accuracy: 0.9924 - val_loss: 1.0997 - val_accuracy: 0.8157 - lr: 5.0000e-09
Epoch 6/100
1614/6093 [======>.......................] - ETA: 53:32 - loss: 0.0250 - accuracy: 0.9918

KeyboardInterrupt: 

## Saving weights and testing on a Single Image

In [19]:
#siamese_model.save_weights('Weights/Siamese_Model_DMR')

In [72]:
import cv2
img = cv2.imread('mp3.jpeg')
img = cv2.resize(img, (224, 224))
img = img / 255
img = img[np.newaxis, ]
pred = siamese_model.predict(img)
pred = tf.keras.layers.Activation('softmax')(pred)

1/1 [==============================] - 0s 136ms/step


In [73]:
pred

<tf.Tensor: shape=(1, 250), dtype=float32, numpy=
array([[3.36921949e-11, 2.55834243e-10, 1.81411215e-14, 8.55646283e-13,
        1.93425134e-08, 3.17666036e-06, 3.83039867e-12, 1.35907621e-05,
        2.85955455e-08, 6.24926721e-10, 8.46714365e-12, 3.70955877e-14,
        3.16155463e-10, 1.21035628e-05, 4.51122389e-07, 9.51008445e-08,
        7.85020432e-13, 2.35234256e-06, 9.75206096e-11, 1.51906896e-08,
        3.56956846e-08, 8.72539454e-12, 9.52912445e-13, 2.37324301e-07,
        3.79681009e-11, 1.92108936e-15, 2.21462804e-09, 7.00598865e-15,
        1.01656518e-07, 1.36973211e-10, 1.87118451e-06, 3.36624417e-08,
        4.36106307e-09, 2.11546535e-07, 4.64958077e-07, 1.58030339e-10,
        2.46378229e-09, 8.15796142e-08, 4.09631147e-15, 6.72274769e-09,
        7.82948191e-05, 7.86694043e-11, 2.32829245e-13, 2.80384587e-11,
        2.34440475e-14, 1.00779836e-11, 3.36748126e-13, 5.45969087e-05,
        5.29063549e-08, 2.52537635e-09, 6.96688929e-08, 6.37536746e-10,
        1.8922

In [74]:
np.argmax(pred, axis = 1)

array([152], dtype=int64)

## Testing on the validation set to see accuracy scores for different batches in the Validation Set

In [15]:
accs = []
preds = []
labels_ = []
siamese_model.load_weights('Weights/Siamese_Model_DMR')
i = 0
for images, labels in valid_set:
    if i == 50:
        break
    pred = siamese_model.predict(images)
    pred = tf.keras.layers.Activation('softmax')(pred)
    accs.append(accuracy_score(np.argmax(labels, axis = 1), np.argmax(pred, axis = 1)))
    labels_.append(labels)
    preds.append(pred)
    i += 1

1/1 [==============================] - 0s 160ms/step


In [16]:
np.argmax(labels_[18], axis = 1),  np.argmax(preds[18], axis = 1)

(array([ 43, 203,  85,  21, 175, 174,  68,  62,  16, 229,  97, 106,   3,
         90, 236, 249], dtype=int64),
 array([131, 203,  85,  56, 175, 201, 206, 156,  16, 229,  97, 110,   3,
         90,  11, 249], dtype=int64))

In [17]:
accs

[0.5625,
 0.9375,
 0.75,
 0.875,
 0.6875,
 0.9375,
 0.875,
 0.9375,
 0.8125,
 0.875,
 0.9375,
 0.8125,
 0.875,
 1.0,
 0.75,
 0.75,
 0.6875,
 0.8125,
 0.5625,
 0.75,
 0.875,
 0.75,
 0.875,
 0.8125,
 0.75,
 0.75,
 0.9375,
 0.875,
 0.9375,
 0.875,
 0.8125,
 0.625,
 0.875,
 0.9375,
 0.75,
 0.8125,
 0.5625,
 0.8125,
 0.9375,
 0.8125,
 0.75,
 0.8125,
 0.875,
 0.8125,
 0.9375,
 0.6875,
 0.875,
 0.625,
 0.8125,
 0.8125]

## Average Validation Accuracy found out to be 82.125%

In [18]:
avg = np.average(accs) * 100
avg

81.125

## Utility Function for Testing

## Make sure the directory structure is as follows. Dataset->Test->{Test Images}

In [12]:
def load_data_test(csv_dir, mode = 'Test'):
    df = pd.read_csv(csv_dir, header = None)
    df[1] = (df[1] - 1).astype(str)
    df[0] = df[0].astype(str)
    df[0] = 'Dataset/' + mode + '/' + df[0] + '.png'
    df.columns = ['filename', 'class']
        
    return df

## Testing Function

In [13]:
def return_test_accuracy(test_csv_dir):
    df_test = load_data_test(test_csv_dir)
    image_gen = ImageDataGenerator(
        rescale = 1. / 255,
    )
    test_set = image_gen.flow_from_dataframe(
        df_test,
        target_size = (224, 224),
        batch_size = 16,
        class_mode = 'categorical'
    )
    siamese_model.load_weights('Weights/Siamese_Model_DMR')
    accs = []
    for images, labels in test_set:
        pred = siamese_model.predict(images)
        pred = tf.keras.layers.Activation('softmax')(pred)
        accs.append(accuracy_score(np.argmax(labels, axis = 1), np.argmax(pred, axis = 1)))
    return np.average(accs) * 100

In [28]:
def return_csv_accuracy_per_image(images_path):
    image_names = []
    predicted_labels = []
    siamese_model.load_weights('Weights/Siamese_Model_DMR')

    for image_file in os.listdir(images_path):
        image_path = os.path.join(images_path, image_file)
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))
        image = image / 255.0
        image = image[np.newaxis, ]
        prediction = siamese_model.predict(image)
        prediction = tf.keras.layers.Activation('softmax')(prediction)
        predicted_label = np.argmax(prediction, axis = 1)
        image_names.append(image_file)
        predicted_labels.append(predicted_label[0] + 1)
    
    df = pd.DataFrame({'Image_Name': image_names, 'Predicted_Label': predicted_labels})
    csv_file_path = 'Test_Predicted.csv'
    df.to_csv(csv_file_path, index = False)
    
    print(f"CSV file saved at: {csv_file_path}")

In [ ]:
print('The test accuracy is', return_test_accuracy('test_csv_dir'))

In [17]:
return_csv_accuracy_per_image('Dataset_Test/Test')

1/1 [==============================] - 0s 143ms/step


1/1 [==============================] - 0s 137ms/step


1/1 [==============================] - 0s 137ms/step


1/1 [==============================] - 0s 134ms/step


1/1 [==============================] - 0s 136ms/step


...

1/1 [==============================] - 0s 202ms/step


1/1 [==============================] - 0s 188ms/step


1/1 [==============================] - 0s 163ms/step


1/1 [==============================] - 0s 174ms/step


1/1 [==============================] - 0s 171ms/step
CSV file saved at: Test_Predicted.csv
